# Notebook 5 — Output Parsers y Structured Outputs
**Clase 2: Ingeniería de Prompts** | IA Generativa

## Objetivos
- Usar `StrOutputParser`, `JsonOutputParser` y `PydanticOutputParser`
- Incluir instrucciones de formato en el prompt automáticamente
- Validar y estructurar respuestas con Pydantic
- Manejar errores de parseo con `OutputFixingParser`

---

In [1]:
#!pip install langchain langchain-google-genai python-dotenv pydantic -q

In [ ]:
import os
import json
from typing import List, Optional

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain.output_parsers import PydanticOutputParser, OutputFixingParser

from google.colab import userdata
API_KEY = userdata.get('GEMINI_API_KEY')

MODEL = "gemini-2.5-flash-lite"

llm = ChatGoogleGenerativeAI(model=MODEL, temperature=0.5, google_api_key=API_KEY)

print("✅ Entorno listo")

## 1. StrOutputParser — El más simple

In [ ]:
# Sin parser: devuelve AIMessage
resp_raw = llm.invoke("Di hola en 3 idiomas.")
print(f"Sin parser → tipo: {type(resp_raw).__name__}")
print(f"  resp_raw.content: {resp_raw.content}")

# Con StrOutputParser: devuelve string directamente
cadena = ChatPromptTemplate.from_messages([
    ("human", "{input}")
]) | llm | StrOutputParser()

resp_str = cadena.invoke({"input": "Di hola en 3 idiomas."})
print(f"\nCon StrOutputParser → tipo: {type(resp_str).__name__}")
print(f"  valor: {resp_str}")

## 2. JsonOutputParser — Extraer JSON de la respuesta

In [ ]:
# Sin schema: parsea cualquier JSON que el modelo devuelva
parser_json = JsonOutputParser()

template_json = ChatPromptTemplate.from_messages([
    ("system", "Siempre responde con un JSON válido sin markdown ni backticks."),
    ("human", "{pregunta}")
])

cadena_json = template_json | llm | parser_json

resultado = cadena_json.invoke({
    "pregunta": "Dame información sobre Madrid: población, país, fundación y famoso por qué."
})

print(f"Tipo resultado: {type(resultado).__name__}")
print("\nContenido:")
print(json.dumps(resultado, ensure_ascii=False, indent=2))

In [ ]:
# Problema frecuente: el modelo añade markdown al JSON
# ¡El JsonOutputParser de LangChain lo maneja automáticamente!
template_con_markdown = ChatPromptTemplate.from_messages([
    ("human", "Dame información básica sobre Python (el lenguaje) en formato JSON")
])

# Aunque el modelo responda con ```json ... ```, el parser lo limpia
try:
    resultado = (template_con_markdown | llm | parser_json).invoke({})
    print("✅ Parseado correctamente:")
    print(json.dumps(resultado, ensure_ascii=False, indent=2))
except Exception as e:
    print(f"❌ Error de parseo: {e}")

## 3. Pydantic — Definir el schema exacto de la respuesta

In [ ]:
# Definir el schema con Pydantic
class AnalisisSentimiento(BaseModel):
    clasificacion: str = Field(
        description="Clasificación del sentimiento: POSITIVO, NEGATIVO o MIXTO"
    )
    puntuacion: int = Field(
        description="Puntuación del 1 al 5, donde 1=muy negativo y 5=muy positivo",
        ge=1, le=5  # greater_equal=1, less_equal=5
    )
    aspectos_positivos: List[str] = Field(
        description="Lista de aspectos positivos mencionados en la reseña"
    )
    aspectos_negativos: List[str] = Field(
        description="Lista de aspectos negativos mencionados en la reseña"
    )
    recomendaria: bool = Field(
        description="Si el cliente recomendaría el producto basándose en la reseña"
    )
    resumen: str = Field(
        description="Resumen breve de la reseña en máximo 20 palabras"
    )

    @field_validator('clasificacion')
    @classmethod
    def validar_clasificacion(cls, v):
        permitidos = {'POSITIVO', 'NEGATIVO', 'MIXTO'}
        if v.upper() not in permitidos:
            raise ValueError(f"Clasificación debe ser una de: {permitidos}")
        return v.upper()

# Crear el parser Pydantic
parser_pydantic = PydanticOutputParser(pydantic_object=AnalisisSentimiento)

# Ver las instrucciones de formato que genera automáticamente
print("Instrucciones de formato generadas por el parser:")
print("-" * 50)
print(parser_pydantic.get_format_instructions())

In [ ]:
# Template que incluye las instrucciones de formato automáticamente
template_pydantic = ChatPromptTemplate.from_messages([
    ("system", """Eres un analizador de reseñas de productos.

{format_instructions}"""),
    ("human", "Analiza esta reseña:\n\n{resena}")
])

cadena_pydantic = template_pydantic | llm | parser_pydantic

RESENA_COMPLEJA = """
Compré este robot de cocina hace 3 meses y tengo sentimientos encontrados.
La potencia es brutal y pica verduras perfectamente, pero el ruido es ensordecedor.
El material parece de calidad y el diseño es elegante. Sin embargo, la app es un desastre
absoluto, se cuelga constantemente. El servicio técnico fue rapidísimo cuando reporté el problema.
Por el precio (450€) esperaba más, especialmente en software. El hardware, un 9; el software, un 3.
"""

resultado = cadena_pydantic.invoke({
    "resena": RESENA_COMPLEJA,
    "format_instructions": parser_pydantic.get_format_instructions()
})

print(f"Tipo resultado: {type(resultado).__name__}")
print(f"\nClasificación: {resultado.clasificacion}")
print(f"Puntuación: {resultado.puntuacion}/5")
print(f"¿Recomendaría?: {'Sí' if resultado.recomendaria else 'No'}")
print(f"\nAspectos positivos:")
for asp in resultado.aspectos_positivos:
    print(f"  ✅ {asp}")
print(f"\nAspectos negativos:")
for asp in resultado.aspectos_negativos:
    print(f"  ❌ {asp}")
print(f"\nResumen: {resultado.resumen}")

## 4. Schemas más complejos con Pydantic

In [ ]:
# Schema anidado para extracción de información estructurada
class Persona(BaseModel):
    nombre: str = Field(description="Nombre completo de la persona")
    edad: Optional[int] = Field(default=None, description="Edad en años, si se menciona")
    ocupacion: Optional[str] = Field(default=None, description="Ocupación o cargo")

class Empresa(BaseModel):
    nombre: str = Field(description="Nombre de la empresa")
    sector: str = Field(description="Sector o industria de la empresa")
    pais: Optional[str] = Field(default=None, description="País donde opera")

class ExtraccionNoticia(BaseModel):
    titulo_sugerido: str = Field(description="Título breve sugerido para la noticia")
    personas: List[Persona] = Field(description="Personas mencionadas en el texto")
    empresas: List[Empresa] = Field(description="Empresas mencionadas en el texto")
    fecha_evento: Optional[str] = Field(default=None, description="Fecha del evento si se menciona")
    cifras_clave: List[str] = Field(description="Cifras o datos numéricos importantes")
    palabras_clave: List[str] = Field(description="5 palabras clave del texto")

parser_noticia = PydanticOutputParser(pydantic_object=ExtraccionNoticia)

template_extraccion = ChatPromptTemplate.from_messages([
    ("system", "Extrae información estructurada del texto. {format_instructions}"),
    ("human", "{texto}")
])

cadena_extraccion = template_extraccion | llm | parser_noticia

NOTICIA = """
La startup madrileña DataCore, fundada en 2022 por María López (CEO, 34 años) y
Javier Sánchez (CTO, 38 años), ha cerrado una ronda Serie A de 8 millones de euros
liderada por el fondo alemán TechVentures. La empresa, especializada en análisis
de datos para el sector retail, cuenta ya con 45 empleados y espera alcanzar los
3 millones de euros de ARR a finales de 2024. El capital se destinará principalmente
a la expansión internacional hacia Francia e Italia.
"""

resultado = cadena_extraccion.invoke({
    "texto": NOTICIA,
    "format_instructions": parser_noticia.get_format_instructions()
})

print(f"📰 Título: {resultado.titulo_sugerido}")
print(f"\n👥 Personas:")
for p in resultado.personas:
    edad_str = f", {p.edad} años" if p.edad else ""
    print(f"  • {p.nombre}{edad_str} — {p.ocupacion}")
print(f"\n🏢 Empresas:")
for e in resultado.empresas:
    print(f"  • {e.nombre} ({e.sector}, {e.pais})")
print(f"\n📊 Cifras clave: {', '.join(resultado.cifras_clave)}")
print(f"🔑 Palabras clave: {', '.join(resultado.palabras_clave)}")

## 5. OutputFixingParser — Reparar respuestas inválidas

In [ ]:
# Simular una respuesta inválida del modelo
respuesta_mal_formateada = """
Aquí está mi análisis:

- clasificacion: positivo (con minúscula y sin comillas)
- puntuacion: "cuatro" (string en lugar de int)
- aspectos_positivos: [buena calidad, envío rápido]
- aspectos_negativos: ninguno
- recomendaria: sí
- resumen: producto muy bueno
"""

print("Intentando parsear respuesta mal formateada...")
try:
    resultado_directo = parser_pydantic.parse(respuesta_mal_formateada)
    print("✅ Parseado OK")
except Exception as e:
    print(f"❌ Error esperado: {type(e).__name__}")
    print(f"   {str(e)[:200]}")

In [ ]:
# OutputFixingParser: usa el LLM para corregir automáticamente
fixing_parser = OutputFixingParser.from_llm(
    parser=parser_pydantic,
    llm=llm
)

print("Intentando reparar con OutputFixingParser...")
try:
    resultado_corregido = fixing_parser.parse(respuesta_mal_formateada)
    print(f"✅ Reparado correctamente")
    print(f"   Clasificación: {resultado_corregido.clasificacion}")
    print(f"   Puntuación: {resultado_corregido.puntuacion}")
    print(f"   Recomendaría: {resultado_corregido.recomendaria}")
    print(f"   Resumen: {resultado_corregido.resumen}")
except Exception as e:
    print(f"❌ No pudo repararse: {e}")

## 6. Pipeline completo de producción

Combinamos todo en una función robusta de producción:

In [ ]:
class ResultadoAnalisis(BaseModel):
    """Schema de salida para el análisis de reseñas."""
    clasificacion: str = Field(description="POSITIVO, NEGATIVO o MIXTO")
    puntuacion: int = Field(description="Puntuación 1-5", ge=1, le=5)
    resumen: str = Field(description="Resumen en máx 15 palabras")
    accion_recomendada: str = Field(
        description="Acción recomendada para el equipo: responder_agradeciendo, responder_disculpandose, escalar_soporte, monitorizar"
    )


def analizar_resena_produccion(resena: str, reintentos: int = 2) -> dict:
    """
    Pipeline robusto de análisis de reseñas.
    Usa OutputFixingParser como fallback si el parseo falla.
    """
    parser = PydanticOutputParser(pydantic_object=ResultadoAnalisis)
    fixing_parser = OutputFixingParser.from_llm(parser=parser, llm=llm)

    template = ChatPromptTemplate.from_messages([
        ("system", """
Eres un sistema de análisis de reseñas de e-commerce.
Analiza cada reseña y proporciona un análisis estructurado.

{format_instructions}

Para accion_recomendada:
- responder_agradeciendo: reseñas positivas (4-5 estrellas)
- responder_disculpandose: reseñas negativas (1-2 estrellas)
- escalar_soporte: menciona problemas técnicos o legales
- monitorizar: reseñas mixtas o ambiguas
"""),
        ("human", "Analiza: {resena}")
    ])

    for intento in range(reintentos + 1):
        try:
            # Intento principal
            cadena = template | llm | parser
            resultado = cadena.invoke({
                "resena": resena,
                "format_instructions": parser.get_format_instructions()
            })
            return {"status": "ok", "datos": resultado.dict(), "intento": intento + 1}

        except Exception as e:
            if intento < reintentos:
                print(f"⚠️  Intento {intento+1} fallido, intentando con fixing parser...")
                try:
                    # Fallback con fixing parser
                    raw = (template | llm | StrOutputParser()).invoke({
                        "resena": resena,
                        "format_instructions": parser.get_format_instructions()
                    })
                    resultado = fixing_parser.parse(raw)
                    return {"status": "fixed", "datos": resultado.dict(), "intento": intento + 1}
                except:
                    pass
            else:
                return {"status": "error", "error": str(e), "intento": intento + 1}


# Test con varias reseñas
resenas_test = [
    "Producto excelente, llegó en 24h y superó mis expectativas. ¡Lo recomiendo!",
    "Llevo 2 semanas esperando y nadie responde a mis emails. Voy a ir a consumidores.",
    "La calidad del producto es buena pero el tiempo de entrega fue el doble de lo prometido.",
]

print("PIPELINE DE ANÁLISIS EN PRODUCCIÓN")
print("=" * 60)

for i, resena in enumerate(resenas_test, 1):
    print(f"\n[{i}] '{resena[:60]}...'")
    resultado = analizar_resena_produccion(resena)

    if resultado["status"] in ("ok", "fixed"):
        datos = resultado["datos"]
        print(f"    🏷️  {datos['clasificacion']} | ⭐ {datos['puntuacion']}/5")
        print(f"    📋 {datos['resumen']}")
        print(f"    ⚡ Acción: {datos['accion_recomendada']}")
        if resultado["status"] == "fixed":
            print(f"    ⚠️  Requirió corrección automática")
    else:
        print(f"    ❌ Error: {resultado['error']}")

## 7. Bonus: Generar el schema desde el propio modelo

In [ ]:
# Ver el JSON Schema que genera Pydantic (útil para documentación y APIs)
import json

print("JSON Schema de ResultadoAnalisis:")
print(json.dumps(ResultadoAnalisis.model_json_schema(), indent=2, ensure_ascii=False))

## Resumen

| Parser | Cuándo usarlo | Tipo de salida |
|---|---|---|
| `StrOutputParser` | Siempre (caso base) | `str` |
| `JsonOutputParser` | JSON sin schema estricto | `dict` / `list` |
| `PydanticOutputParser` | Schema exacto con validación | Instancia Pydantic |
| `OutputFixingParser` | Como fallback ante errores | Igual que el parser base |

### Flujo recomendado en producción:
```
Definir schema Pydantic
       ↓
Crear PydanticOutputParser → genera format_instructions
       ↓
Incluir format_instructions en el template
       ↓
Cadena: template | llm | parser
       ↓
Fallback: OutputFixingParser si hay error
```

**➡️ Siguiente:** Notebook 6 — LCEL: encadenando prompts

### 🏋️ Ejercicios
1. Crea un schema Pydantic para extraer ingredientes de una receta: nombre, cantidad, unidad y es_opcional
2. Añade un validador que garantice que `puntuacion` solo acepta 1, 2, 3, 4 o 5 (no decimales)
3. Prueba el `OutputFixingParser` pasándole deliberadamente una respuesta mal formateada y verifica que la corrige